**Record of a July 2026 Colab run.** This notebook keeps the original outputs, though its first cell was replaced to clone this repository, so those outputs came from the code of the time. To reproduce the result today, use [`docs/reproducing.md`](../docs/reproducing.md).

# Phase 5 — decoder-aligned full-group GRPO training

This run performs 60 optimizer steps with one complete eight-completion GRPO group per step (480 completions total). Formatting curriculum weight reaches zero after 40 groups. Frozen and trained policies are evaluated with 50 paired stochastic samples using the same decoder as training; greedy output is reported separately. `click-test` is one seed-invariant prompt, so this measures fixed-task format alignment rather than held-out generalization.

## 1. Fetch the repository

In [ ]:
# Fetch the public repository.
import os
import subprocess
from pathlib import Path

repository_path = Path('/content/browser-control')
if not repository_path.exists():
    subprocess.run(['git', 'clone', '--branch', 'lfm-browser-control',
                    'https://github.com/DerrickKirimi/browser-control.git', str(repository_path)], check=True)
os.chdir(repository_path)


## 2. Install without replacing Colab's PyTorch

In [2]:
from importlib.metadata import version

torch_before = version('torch')
%pip install -q -e ".[environment,model,training]" pytest
torch_after = version('torch')
print('PyTorch:', torch_before, '->', torch_after)
if torch_after != torch_before:
    raise RuntimeError('Project installation replaced Colab PyTorch')
%pip uninstall -y torchao
!python -m playwright install --with-deps chromium

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.6/191.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 137.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Validate GPU, decoder parity, and pinned TRL batching

In [3]:
!python scripts/check_colab_gpu.py
!python -m pytest -q tests/test_trl_contract.py tests/test_training.py tests/test_transformers_policy.py

{
  "cudnn_version": 91900,
  "device": {
    "compute_capability": "7.5",
    "index": 0,
    "name": "Tesla T4",
    "supports_bfloat16": false,
    "total_memory_bytes": 15637086208,
    "training_dtype": "float16"
  },
  "torch_cuda_version": "12.8",
  "torch_version": "2.11.0+cu128"
}
......................                                                   [100%]


## 4. Run 60 full-group optimizer steps

In [4]:
!python scripts/train_controlled.py --seed 23 --max-steps 60 --curriculum-steps 40 --num-generations 8 --per-device-train-batch-size 8 --generation-batch-size 8 --output-dir artifacts/phase5/full-group-seed-23

2026-07-22 10:23:19.704550: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 91.6kB [00:00, 202MB/s]
tokenizer.json: 4.73MB [00:00, 167MB/s]
special_tokens_map.json: 100% 434/434 [00:00<00:00, 4.16MB/s]
chat_template.jinja: 1.38kB [00:00, 8.54MB/s]
config.json: 100% 999/999 [00:00<00:00, 10.0MB/s]
model.safetensors: 100% 709M/709M [12:20<00:00, 957kB/s]
generation_config.json: 100% 137/137 [00:00<00:00, 1.06MB/s]
  0% 0/60 [00:00<?, ?it/s]Could not estimate the number of tokens of the input, floating-point operations will not be computed
{'loss': -0.3003, 'grad_norm': 3.322187662124634, 'learning_rate': 0.0001, 'num_tokens': 1220.0, 'completions/mean_length': 15.5, 'completions/min_length': 6.0, 'completions/max_length': 32.

## 5. Inspect and download the report and adapter

In [5]:
import json
import shutil
from pathlib import Path

from google.colab import files

output_dir = Path('artifacts/phase5/full-group-seed-23')
run_stem = 'phase5-click-test-seed-0023-groups-0060-generations-0008'
report_path = output_dir / f'{run_stem}-report.json'
status = json.loads((output_dir / 'run-status.json').read_text())
if status.get('status') != 'success':
    if report_path.is_file():
        files.download(str(report_path))
    raise RuntimeError(f'Training did not succeed: {status}')
report = json.loads(report_path.read_text())
print('batching:', report['generation_batching'])
print('frozen stochastic:', report['frozen_evaluation']['stochastic']['successes'], '/ 50')
print('trained stochastic:', report['trained_evaluation']['stochastic']['successes'], '/ 50')
print('success-rate delta:', report['strict_success_delta'])
archive = shutil.make_archive(f'{run_stem}-adapter', 'zip', output_dir / 'adapter')
files.download(str(report_path))
files.download(archive)

batching: {'generation_batch_size': 8, 'gradient_accumulation_steps': 1, 'num_generations': 8, 'per_device_train_batch_size': 8, 'steps_per_generation': 1}
frozen stochastic: 2 / 50
trained stochastic: 50 / 50
success-rate delta: 0.96


[browser download widget removed]

[browser download widget removed]

[browser download widget removed]

[browser download widget removed]